# Simulation scripts {#sec-simulation-scripts}

Here, we provide the scripts to simulate the learning behavior of the agents in the model environment. We start by importing the required packages and defining some configurations.

In [1]:
 #| default_exp SimulationScripts

In [2]:
 #| export
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from _code.UncertainDecisionProblem import \
    SingleAgentUncertainDecisionProblem, TwoAgentUncertainDecisionProblem
from _code.LearningDynamics import LearningAgents

from skopt.sampler import Lhs
from skopt.space import Space

import hashlib
from collections import namedtuple

To avoid issues with floating point arithmetic, we refine the `linspace` function from numpy.

In [3]:
 #| export
def create_consistent_linspace(start, stop, number, decimals=8):
    return np.round(np.linspace(start, stop, number, dtype=np.float64), decimals)

## Reward outperformance measure

We define the reward outperformance, a measure of collective intelligence, if you will, as the difference between the two-agent reward and the single-agent reward, normalized by the single-agent reward.

In [4]:
 #| export
def collective_intelligence(multiagent_reward, 
                            singleagent_reward):
    return (multiagent_reward - singleagent_reward) / singleagent_reward

## Initial strategies

We use latin hypercube sampling to create the initial strategies.

In [5]:
 #| export
def create_inital_lhs(maei, # multiagent environment interface
                      number, # number of initial policies to generate
                      iterations=1000,  # number of interations used for latin hypercube sampling
                      seed=42): # seed for random number generator
    """
    Returns near random samples of initial policies
    """
    assert maei.M == 2, 'Sampling for M>2 not straightforward'
    # https://www.egr.msu.edu/~kdeb/papers/c2018010.pdf
    # https://www.cs.cmu.edu/~nasmith/papers/smith+tromble.tr04.pdf
    
    eps = 10**(-6)
    space = Space(maei.N * maei.Q * (maei.M-1)*[(0.0+eps, 1.0-eps)])
    
    # generate latin hyper cubes
    lhs = Lhs(criterion="maximin", iterations=iterations)
    x = lhs.generate(space.dimensions, number, random_state=seed)
    x = np.array(x).reshape(number, maei.N, maei.Q, maei.M-1)
    
    # complete and normalize
    inits = np.zeros((number, maei.N, maei.Q, maei.M))
    inits[..., 0] = x[...,0]
    inits[..., 1] = 1 - x[...,0]
    
    return inits

### Example latin hyper cube sampling

We test the latin hyper cube sample for 4 initial strategies at the single-agent decision problem. For each state $s$, we find exactly one strategy with $\frac{i}{4}<X_{0,s,0}\leq\frac{i+1}{4}$ for each $0\leq i < 4$.

In [6]:
#| fig-cap: Example of the initial strategies as latin hypercube samples.
number = 4
env = SingleAgentUncertainDecisionProblem(noiselevel=0.5)
maei = LearningAgents(env, learning_rates=0.25, discount_factors=0.9)
Xkioa = create_inital_lhs(maei, number)

fig, axes = plt.subplots(1,4, figsize=(14,3))
def generate_colors(number): return cm.rainbow(np.linspace(0, 1, number))

for j, ax in enumerate(axes):
    probs = []
    for i in range(0, number):
        probs.append(Xkioa[i][0][j][0])  #agent 0 action 0 under observation 0
        ax.axhline(y=i/number, color='gray', linestyle='-', linewidth=0.5)
    
    ax.axhline(y=1.0, color='gray', linestyle='-', linewidth=0.5) 
    ax.scatter(np.arange(number), probs, color=generate_colors(number),
               marker='o', s=50)
    ax.set_ylabel(f'X(s={j})')
plt.tight_layout()

<Figure size 4200x900 with 4 Axes>

## Learning trajectories

### Compute

First, we create a function to compute the policy trajectory for a given set of policies.

In [7]:
 #| export
def compute_trajectories(maei,        # multiagent environment interface
                         inits,       # initial policies
                         maxT=1000,   # maximal number of learning steps
                         tolerance=10e-7): # tolerance for convergence
    """
    Returns array of policy trajectories for different initial policies and 
    array of whether they converged
    """
    trjs = []; fprs = []
    leni = len(inits)
    
    for xi, x0 in enumerate(inits):
        print("\r [Computing policy trajectories]:",
              np.round(xi/leni, 4), end='')
        x = x0.copy()
        
        trj, fpr = maei.trajectory(x, Tmax=maxT, tolerance=tolerance)
        
        trjs.append(trj.astype(np.float32))
        fprs.append(fpr)
    
    print("")
    return np.array(trjs, dtype=object), np.array(fprs)

Testing the function,

In [8]:
number = 7
env = TwoAgentUncertainDecisionProblem(noiselevel=0.5)
# maei = POstratAC(env, learning_rates=0.25, discount_factors=0.9)

maei = LearningAgents(env, learning_rates=0.25)
Xkioa = create_inital_lhs(maei, number, seed=5)

timeseries_Xktioa, fprs = compute_trajectories(maei, Xkioa, maxT=1000)

print("\nLengths of timeseries: ", [len(Xtioa) for Xtioa in timeseries_Xktioa])

 [Computing policy trajectories]: 0.1429

 [Computing policy trajectories]: 0.7143

 [Computing policy trajectories]: 0.8571

Lengths of timeseries:  [1000, 1000, 1000, 1000, 1000, 999, 949]


Let us create a small function the check the output of the function `compute_trajectory` for a given set of policies.

In [9]:
 #| export
def check_run(trjs, fprs=None):
    """
    plots histogram of trajectory length after computing
    """
    #if fprs is not None:
        #print('Unique fixed points reached:', np.unique(fprs))
    plt.hist([len(traj) for traj in trjs], bins=20);
    plt.title('Histrogram of trajectories lengths')


In [10]:
#| fig-cap: Histogram of learning time steps required for a set of different initial strategies.
check_run(timeseries_Xktioa, fprs)

<Figure size 2700x2100 with 1 Axes>

To save the computed runs to a re-identifaible file, we create a function `_transform_tensor_into_hash` which transforms an inital policy tensor into a hash.

In [11]:
 #| export   
def _transform_tensor_into_hash(tens):
    """Transform tens into a string for filename saving"""
    r = int(hashlib.sha512(str(tens).encode('utf-8')).hexdigest()[:16], 16)
    return r

For example,

In [12]:
_transform_tensor_into_hash(Xkioa)

10980271089415640880

### Obtain
We summarize these computation functions into a `obtain_trajectories` function.


In [13]:
 #| export   
def obtain_trajectories(maei,  # multiagent environment interface
                        initXkioa,  # initial policies
                        verbose=True,  # indicates if explanations should be printed
                        maxT=5000,  # maximal number of learning steps
                        ddir='data'):  # directory where to save data
    """
    Loads or computes and saves learning trajectories.
    """
    fn = ddir + '/TRAJECTORIES_' + maei.id() + '_'\
        + str(_transform_tensor_into_hash(initXkioa))
    fn += ".npz"
    
    try:
        dat = np.load(fn, allow_pickle=True)
        ddic = dict(zip((k for k in dat), (dat[k] for k in dat)))
        print("Loading ", fn) if verbose else None
    
    except:
        print("Computing ", fn) if verbose else None
        trjs, fprs = compute_trajectories(maei, initXkioa, maxT=maxT)
        check_run(trjs, fprs)
        # rtrajs = obtain_rewards(AEi, πtrajs)§
        
        ddic = dict(trjs=trjs, fprs=fprs)
        np.savez_compressed(fn, **ddic)
        dat = np.load(fn, allow_pickle=True)
        ddic = dict(zip((k for k in dat), (dat[k] for k in dat)))
    
    return ddic['trjs'], ddic['fprs']

Testing the function, first when there is prior file.

In [14]:
#| output: false
timeseries_Xktioa, fprs = obtain_trajectories(maei, Xkioa, maxT=1000, ddir='.')

Computing  ./TRAJECTORIES_TwoAgentUncertainDecisionProblem_0.5_0.5_1__jLearningAgents_PartObs_[0.25 0.25]_10980271089415640880.npz
 [Computing policy trajectories]: 0.0

 [Computing policy trajectories]: 0.2857

 [Computing policy trajectories]: 0.5714

 [Computing policy trajectories]: 0.8571

<Figure size 2700x2100 with 1 Axes>

When the file exists, it is not re-comuted.

In [15]:
#| output: asis
print("\\begin{OutputCode}") #| hide_line
timeseries_Xktioa, fprs = obtain_trajectories(maei, Xkioa, maxT=1000, ddir='.')
print("\\end{OutputCode}") #| hide_line

\begin{OutputCode}
Loading  ./TRAJECTORIES_TwoAgentUncertainDecisionProblem_0.5_0.5_1__jLearningAgents_PartObs_[0.25 0.25]_10980271089415640880.npz
\end{OutputCode}


We delete the data, since this was only to test the function. In later use, we will give a more pronounced data directory.

In [16]:
#| output: false
!rm *.npz

/opt/homebrew/Caskroom/miniforge/base/envs/iw/lib/python3.14/pty.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


### Final strategies

We also want to study the final strategies that the agents learn.

In [17]:
 #| export
def final_policies_Xkioa(Xktioa):
    return np.array([Xtioa[-1] for Xtioa in Xktioa])

Testing,

In [18]:
final_Xkioa = final_policies_Xkioa(timeseries_Xktioa)
final_Xkioa.shape

(7, 2, 13, 2)

### Final rewards

Last, we create a function to compute the final rewards of a set of learning trajectories.

In [19]:
 #| export
def final_rewards_Rki(maei,  # multiagent environment interface
                      Xktioa): # learning trajectories
    """
    Returns the rewards `Rki` that agents receive at the end of the learning 
    trajectories `Xktioa`.
    """
    if not maei.has_last_obsdist:    # to avoid a small bug 
        maei.obsdist(Xktioa[0][-1].astype(np.float32))  # with jax' just-in-time compilation
    
    Rki = []
    for Xtioa in Xktioa:
        Xioa = Xtioa[-1]
        Ri = maei.Ri(Xioa.astype(np.float32))
        Rki.append(Ri)
        
    return np.array(Rki)

Testing the function,

In [20]:
final_rewards_Rki(maei, timeseries_Xktioa)

array([[0.6862721 , 0.68627197],
       [0.6862698 , 0.68627214],
       [0.68627185, 0.68627137],
       [0.68627226, 0.68627185],
       [0.68627214, 0.68627197],
       [0.68627226, 0.6862722 ],
       [0.6862722 , 0.6862723 ]], dtype=float32)

## Varying noise

Next, we create functions to compute the ensemble runs of the learning trajectories when varying the noise parameter.

### Computing

In [21]:
 #| export
def compute_noise_variation(
    envclass,        # environment class
    noises,          # iterable of noise levels
    envparams,       # dictionary of environment parameters
    agentclass,      # agent class
    agentparams,     # dictionary of agent parameters
    initialXkioa,    # initial policies, k indicates index of initial policy
    maxT=5000,       # maximal number of learning steps
    ddir='data'):    # directory to save and load data
    """Computes the learning trajectories for different noise levels.
       Returns the final rewards and convergence times."""
    
    convergence_times_Tnk = []  # n indicates noise index
    final_rewards_Rnki = []   
    final_policies_Xnkioa = []
    
    for noise in noises:
        print(f" = = = = {noise} = = = =")
        env = envclass(noiselevel=noise, **envparams)
        maei = agentclass(env, **agentparams)
        
        Xktioa, FPRk = obtain_trajectories(
            maei, initialXkioa, verbose=True, maxT=maxT, ddir=ddir)

        convergence_times_Tnk.append([len(Xtios) for Xtios in Xktioa])
        final_rewards_Rnki.append(final_rewards_Rki(maei, Xktioa))
        final_policies_Xnkioa.append(final_policies_Xkioa(Xktioa))
        
        
    return (np.array(convergence_times_Tnk),
            np.array(final_rewards_Rnki),
            np.array(final_policies_Xnkioa))

Testing the function with the single agent environment.

In [22]:
#| output: false
number_of_initial_policies_k = 4
agentclass = LearningAgents
agentparams = dict(learning_rates=0.25, discount_factors=0.9, use_prefactor=False)
envclass = SingleAgentUncertainDecisionProblem
envparams = dict(probabilityA=0.5, likelydistance=1)
env = envclass(noiselevel=0.5)
maei = agentclass(env, **agentparams)
initialXkioa = create_inital_lhs(maei, number_of_initial_policies_k)

noises = [0.001, 0.1, 1.0]

convergence_Tnk, final_Rnki, final_Xnkioa = compute_noise_variation(
    envclass,        # environment class
    noises,          # iterable of noise levels
    envparams,       # dictionary of environment parameters
    agentclass,      # agent class
    agentparams,     # dictionary of agent parameters
    initialXkioa,    # initial policies, k indicates index of initial policy
    ddir='.')


print()
assert convergence_Tnk.shape == (len(noises), number_of_initial_policies_k)
print("Shape of convergence times: ", convergence_Tnk.shape)
print()
assert final_Rnki.shape == (len(noises), number_of_initial_policies_k, maei.N)
print("Shape of final rewards: ", final_Rnki.shape)
print()
assert final_Xnkioa.shape == (len(noises), number_of_initial_policies_k, maei.N, maei.Q, maei.M)
print("Shape of final policies: ", final_Xnkioa.shape)

 = = = = 0.001 = = = =
Computing  ./TRAJECTORIES_SingleAgentUncertainDecisionProblem_0.001_0.5_1__jLearningAgents_PartObs_[0.25]_1518859870959773285.npz
 [Computing policy trajectories]: 0.75


 = = = = 0.1 = = = =
Computing  ./TRAJECTORIES_SingleAgentUncertainDecisionProblem_0.1_0.5_1__jLearningAgents_PartObs_[0.25]_1518859870959773285.npz
 [Computing policy trajectories]: 0.75
 = = = = 1.0 = = = =
Computing  ./TRAJECTORIES_SingleAgentUncertainDecisionProblem_1.0_0.5_1__jLearningAgents_PartObs_[0.25]_1518859870959773285.npz
 [Computing policy trajectories]: 0.0

 [Computing policy trajectories]: 0.75

Shape of convergence times:  (3, 4)

Shape of final rewards:  (3, 4, 1)

Shape of final policies:  (3, 4, 1, 4, 2)


<Figure size 2700x2100 with 1 Axes>

Testing the function with the two agent environment.

In [23]:
#| output: false
number_of_initial_policies_k = 4
agentclass = LearningAgents
agentparams = dict(learning_rates=0.25, discount_factors=0.9, use_prefactor=False)
envclass = TwoAgentUncertainDecisionProblem
envparams = dict(probabilityA=0.5, likelydistance=1)
env = envclass(noiselevel=0.5)
maei = agentclass(env, **agentparams)
initialXkioa = create_inital_lhs(maei, number_of_initial_policies_k)

noises = [0.001, 0.1, 1.0]

convergence_Tnk, final_Rnki, final_Xnkioa = compute_noise_variation(
    envclass,        # environment class
    noises,          # iterable of noise levels
    envparams,       # dictionary of environment parameters
    agentclass,      # agent class
    agentparams,     # dictionary of agent parameters
    initialXkioa,    # initial policies, k indicates index of initial policy
    ddir='.')

print()
assert convergence_Tnk.shape == (len(noises), number_of_initial_policies_k)
print("Shape of convergence times: ", convergence_Tnk.shape)
print()
assert final_Rnki.shape == (len(noises), number_of_initial_policies_k, maei.N)
print("Shape of final rewards: ", final_Rnki.shape)
print()
assert final_Xnkioa.shape == (len(noises), number_of_initial_policies_k, maei.N, maei.Q, maei.M)
print("Shape of final policies: ", final_Xnkioa.shape)

 = = = = 0.001 = = = =
Computing  ./TRAJECTORIES_TwoAgentUncertainDecisionProblem_0.001_0.5_1__jLearningAgents_PartObs_[0.25 0.25]_11824446990828570514.npz
 [Computing policy trajectories]: 0.75
 = = = = 0.1 = = = =


Computing  ./TRAJECTORIES_TwoAgentUncertainDecisionProblem_0.1_0.5_1__jLearningAgents_PartObs_[0.25 0.25]_11824446990828570514.npz
 [Computing policy trajectories]: 0.75
 = = = = 1.0 = = = =
Computing  ./TRAJECTORIES_TwoAgentUncertainDecisionProblem_1.0_0.5_1__jLearningAgents_PartObs_[0.25 0.25]_11824446990828570514.npz
 [Computing policy trajectories]: 0.0

 [Computing policy trajectories]: 0.75

Shape of convergence times:  (3, 4)

Shape of final rewards:  (3, 4, 2)

Shape of final policies:  (3, 4, 2, 13, 2)


<Figure size 2700x2100 with 1 Axes>

We delete the data, since this was only to test the function. In later use, we will give a more pronounced data directory.

In [24]:
#| output: false
!rm *.npz

/opt/homebrew/Caskroom/miniforge/base/envs/iw/lib/python3.14/pty.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


### Obtaining

First, we create a little helper function which transform a dictionary of parameters into a string for the file name.

In [25]:
 #| export
def dict_to_string(paramdic):
    string = ''
    for k, v in paramdic.items():
        string += f'{k}{v}_'
    return string[:-1]

For example, the environmtal parameters,

In [26]:
envparams

{'probabilityA': 0.5, 'likelydistance': 1}

 will be transformed into a string as

In [27]:
dict_to_string(envparams)

'probabilityA0.5_likelydistance1'

We enrich the `obtain_noise_variation` function with a second data directory. One is to retrieve the underlying learning trajectories (`lowlevel_ddir`), the other is to save the ensemble results (`highlevel_ddir`). The latter is to be tracked in the project's git repository, the former not.

In [28]:
 #| export
def obtain_noise_variation(
    envclass,        # environment class
    noises,          # iterable of noise levels
    envparams,       # dictionary of environment parameters
    agentclass,      # agent class
    agentparams,     # dictionary of agent parameters
    initialXkioa,    # initial policies, k indicates index of initial policy
    maxT=5000,       # maximal number of learning steps
    lowlevel_ddir='',  # directory to save and load low-level data
    highlevel_ddir='',  # directory to save and load high-level data
    ):
    """Computes the learning trajectories for different noise levels.
       Returns the final rewards and convergence times."""
    
    # Create file name
    fn = highlevel_ddir + '/NOISEVARIATION_' + envclass.__name__ + '_'\
        + dict_to_string(envparams)\
        + '_' + agentclass.__name__ + '_' + dict_to_string(agentparams)\
        + '_' + "initalX" + str(_transform_tensor_into_hash(initialXkioa))\
        + '_' + "maxT" + str(maxT)\
        + '_' + "Noises" + str(_transform_tensor_into_hash(np.array(noises)))\
        + ".npz"
    
    try:
        dat = np.load(fn, allow_pickle=True)
        ddic = dict(zip((k for k in dat), (dat[k] for k in dat)))
        print("Loading noise variation:", fn)

    except:
        print("Computing noise variation:", fn)
        convergence_Tnk, final_Rnki, final_Xnkioa = compute_noise_variation(
            envclass,        # environment class
            noises,          # iterable of noise levels
            envparams,       # dictionary of environment parameters
            agentclass,      # agent class
            agentparams,     # dictionary of agent parameters
            initialXkioa,    # initial policies, k is index of initial policy
            maxT=maxT,       # maximal number of learning steps
            ddir=lowlevel_ddir)
        ddic = dict(convergence_times_Tnk=convergence_Tnk, 
                    final_rewards_Rnki=final_Rnki,
                    final_policies_Xnkioa=final_Xnkioa)
        np.savez_compressed(fn, **ddic)
        dat = np.load(fn, allow_pickle=True)
        ddic = dict(zip((k for k in dat), (dat[k] for k in dat)))
        
    return (ddic['convergence_times_Tnk'],
            ddic['final_rewards_Rnki'],
            ddic['final_policies_Xnkioa'])

Testing the function with the two agent environment.

In [29]:
!mkdir _datatest

In [30]:
#| output: false
number_of_initial_policies_k = 4
agentclass = LearningAgents
agentparams = dict(learning_rates=0.25, discount_factors=0.9, use_prefactor=False)
envclass = TwoAgentUncertainDecisionProblem
envparams = dict(probabilityA=0.5, likelydistance=1)
env = envclass(noiselevel=0.5)
maei = agentclass(env, **agentparams)
initialXkioa = create_inital_lhs(maei, number_of_initial_policies_k)

noises = [0.001, 0.1, 1.0]

convergence_Tnk, final_Rnki, final_Xnkioa = obtain_noise_variation(
    envclass,        # environment class
    noises,          # iterable of noise levels
    envparams,       # dictionary of environment parameters
    agentclass,      # agent class
    agentparams,     # dictionary of agent parameters
    initialXkioa,    # initial policies, k indicates index of initial policy
    lowlevel_ddir='.',
    highlevel_ddir='_datatest')

print()
assert convergence_Tnk.shape == (len(noises), number_of_initial_policies_k)
print("Shape of convergence times: ", convergence_Tnk.shape)
print()
assert final_Rnki.shape == (len(noises), number_of_initial_policies_k, maei.N)
print("Shape of final rewards: ", final_Rnki.shape)
print()
assert final_Xnkioa.shape == (len(noises), number_of_initial_policies_k, maei.N, maei.Q, maei.M)
print("Shape of final policies: ", final_Xnkioa.shape)

Computing noise variation: _datatest/NOISEVARIATION_TwoAgentUncertainDecisionProblem_probabilityA0.5_likelydistance1_LearningAgents_learning_rates0.25_discount_factors0.9_use_prefactorFalse_initalX11824446990828570514_maxT5000_Noises8009511655927899652.npz
 = = = = 0.001 = = = =
Computing  ./TRAJECTORIES_TwoAgentUncertainDecisionProblem_0.001_0.5_1__jLearningAgents_PartObs_[0.25 0.25]_11824446990828570514.npz
 [Computing policy trajectories]: 0.75


 = = = = 0.1 = = = =
Computing  ./TRAJECTORIES_TwoAgentUncertainDecisionProblem_0.1_0.5_1__jLearningAgents_PartObs_[0.25 0.25]_11824446990828570514.npz
 [Computing policy trajectories]: 0.75


 = = = = 1.0 = = = =
Computing  ./TRAJECTORIES_TwoAgentUncertainDecisionProblem_1.0_0.5_1__jLearningAgents_PartObs_[0.25 0.25]_11824446990828570514.npz
 [Computing policy trajectories]: 0.75



Shape of convergence times:  (3, 4)

Shape of final rewards:  (3, 4, 2)

Shape of final policies:  (3, 4, 2, 13, 2)


<Figure size 2700x2100 with 1 Axes>

Now, data is obtained much more quickly from disk:

In [31]:
#| output: asis
print("\\begin{OutputCode}") #| hide_line
convergence_Tnk, final_Rnki, final_Xnkioa = obtain_noise_variation(
    envclass,        # environment class
    noises,          # iterable of noise levels
    envparams,       # dictionary of environment parameters
    agentclass,      # agent class
    agentparams,     # dictionary of agent parameters
    initialXkioa,    # initial policies, k indicates index of initial policy
    lowlevel_ddir='.',
    highlevel_ddir='_datatest/')

print()
assert convergence_Tnk.shape == (len(noises), number_of_initial_policies_k)
print("Shape of convergence times: ", convergence_Tnk.shape)
print()
assert final_Rnki.shape == (len(noises), number_of_initial_policies_k, maei.N)
print("Shape of final rewards: ", final_Rnki.shape)
print()
assert final_Xnkioa.shape == (len(noises), number_of_initial_policies_k, maei.N, maei.Q, maei.M)
print("Shape of final policies: ", final_Xnkioa.shape)
print("\\end{OutputCode}") #| hide_line

\begin{OutputCode}
Loading noise variation: _datatest//NOISEVARIATION_TwoAgentUncertainDecisionProblem_probabilityA0.5_likelydistance1_LearningAgents_learning_rates0.25_discount_factors0.9_use_prefactorFalse_initalX11824446990828570514_maxT5000_Noises8009511655927899652.npz

Shape of convergence times:  (3, 4)

Shape of final rewards:  (3, 4, 2)

Shape of final policies:  (3, 4, 2, 13, 2)
\end{OutputCode}


Last, we delete the test data.

In [32]:
#| output: false
!rm *.npz

/opt/homebrew/Caskroom/miniforge/base/envs/iw/lib/python3.14/pty.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [33]:
!rm -r _datatest

## Noise variation comparison

Last, we create a function to obtain the comparison between single-agent and two-agent learning performance when varying the noise parameter.

In [34]:
 #| export
def obtain_noise_variation_comparison(lowlevel_ddir, highlevel_ddir,
                                      probabilityA, noise_levels, 
                                      agentclass, maxT,
                                      likelydistance=0.5,
                                      number_of_initial_policies_k=25):
    # Fixed parameters
    agentparams = dict(learning_rates=0.25)
    envparams = dict(probabilityA=probabilityA, likelydistance=likelydistance)

    ## Two-agent model
    envclass = TwoAgentUncertainDecisionProblem
    env = envclass(noiselevel=0.5, **envparams)
    maei = agentclass(env, **agentparams)
    
    if number_of_initial_policies_k == 1:
        initialXkios = np.array([maei.zero_intelligence_policy()])
    else:
        initialXkios = create_inital_lhs(maei, number_of_initial_policies_k)

    twoagent_convergence_Tnk, twoagent_final_Rnki, twoagent_Xnkioa =\
        obtain_noise_variation(
        envclass,        # environment class
        noise_levels,    # iterable of noise levels
        envparams,       # dictionary of environment parameters
        agentclass,      # agent class
        agentparams,     # dictionary of agent parameters
        initialXkios,    # initial policies, k indicates index of initial policy
        maxT=maxT,       # maximal number of learning steps
        lowlevel_ddir=lowlevel_ddir, highlevel_ddir=highlevel_ddir)
        
    # Single-agent model
    envclass = SingleAgentUncertainDecisionProblem
    env = envclass(noiselevel=0.5, **envparams)
    maei = agentclass(env, **agentparams)
    
    if number_of_initial_policies_k == 1:
        initialXkios = np.array([maei.zero_intelligence_policy()])
    else:
        initialXkios = create_inital_lhs(maei, number_of_initial_policies_k)
        
    singleagent_convergence_Tnk, singleagent_final_Rnki, singleagent_Xnkioa =\
        obtain_noise_variation(
        envclass,        # environment class
        noise_levels,    # iterable of noise levels
        envparams,       # dictionary of environment parameters
        agentclass,      # agent class
        agentparams,     # dictionary of agent parameters
        initialXkios,    # initial policies, k indicates index of initial policy
        maxT=maxT,       # maximal number of learning steps
        lowlevel_ddir=lowlevel_ddir, highlevel_ddir=highlevel_ddir)
    
    NoiseVariation = namedtuple('NoiseVariation',
                                ['ta_Tnk', 'ta_Rnki', 'ta_Xnkioa',
                                 'sa_Tnk', 'sa_Rnki', 'sa_Xnkioa'])

    return NoiseVariation(ta_Tnk=twoagent_convergence_Tnk, 
                          ta_Rnki=twoagent_final_Rnki, 
                          ta_Xnkioa=twoagent_Xnkioa,
                          sa_Tnk=singleagent_convergence_Tnk, 
                          sa_Rnki=singleagent_final_Rnki,
                          sa_Xnkioa=singleagent_Xnkioa)

At the very end, we export all functions to the simulation scripts module.

In [35]:
import nbdev
nbdev.export.nb_export("j81_AUX_Simulation.ipynb", "_code")